# Import des librairies

In [9]:
import re
import numpy as np
from dataclasses import dataclass

# Definition classe

In [10]:
@dataclass
class Light:
    col: int
    row: int
    statut: str
    _pos: tuple[int,int] = None


    @property
    def pos(self):
        if self._pos is None:
            self._pos = [self.row, self.col]
        return self._pos

# Nettoyage des données

In [11]:
with open("real.txt") as file:
    grid = np.array([
        [e for e in row]
        for row in file.read().splitlines()
    ])

dim_grid = len(grid)

grid

array([['#', '.', '.', ..., '#', '.', '.'],
       ['#', '#', '#', ..., '#', '.', '#'],
       ['.', '.', '.', ..., '.', '#', '.'],
       ...,
       ['#', '.', '.', ..., '.', '.', '.'],
       ['#', '#', '#', ..., '.', '#', '.'],
       ['.', '#', '.', ..., '#', '#', '.']], shape=(100, 100), dtype='<U1')

# Détermine pour une lumière si on doit la switch

In [12]:
def func_get_new_statut_light(
    in_light: Light,
    in_grid: np.array,
    in_dim_grid: int
):
    n_neighbors = 0

    range_row = range(max(in_light.row-1, 0), min(in_light.row+2, in_dim_grid))
    range_col = range(max(in_light.col-1, 0), min(in_light.col+2, in_dim_grid))

    # Pour chaque voisins
    for row_neighbor in range_row:
        for col_neighbor in range_col:

            # Créer un objet Light
            neighbor = Light(
                col=col_neighbor,
                row=row_neighbor,
                statut=in_grid[row_neighbor,col_neighbor]
            )

            # Ne pas explorer la positon actuelle
            if neighbor.pos == in_light.pos:
                continue
            
            # Si la lumière du voisin est allumée
            if neighbor.statut == "#":

                # On incrémente le compteur
                n_neighbors += 1
    
    # Si la lumière orinale est allumée
    if in_light.statut == "#": 

        # Si le nombre de voisins est compris entre 2 et 3
        if 2 <= n_neighbors <= 3:
            
            # On ne fais rien
            return None

        # Si le nombre de voisins n'est pas compris entre 2 et 3
        else:
            # On l'éteint
            in_light.statut = "."
    
    # Si la lumière originale est éteinte
    else:

        # Si elle a exactement 3 voisins
        if n_neighbors == 3:

            # On l'allume
            in_light.statut = "#"
        
        # Si le nombre de voisins est différent de 3
        else:

            # On ne fais rien
            return None
        
    return in_light


# Récupère les lumières à switch

In [13]:
def func_get_lights_to_switch(
    in_grid: np.array,
    in_dim_grid: int
) -> list[Light]:
    
    lights_to_switch = [
        res
        for num_row, row in enumerate(in_grid)
        for num_col, e in enumerate(row)
        if (res:= func_get_new_statut_light(
                in_dim_grid=in_dim_grid,
                in_grid=in_grid,
                in_light=Light(
                    col= num_col,
                    row = num_row,
                    statut = e
                )
            )) is not None
    ]

    return lights_to_switch

# Switch le statut des lumières récupérés

In [14]:
def func_switch_all_lights_statut(
    in_grid: np.array,
    in_lights_to_switch: list[Light]
) -> np.array:
    
    # Pour chaque lumière à switch
    for light_to_switch in in_lights_to_switch:

        # On switch son statut (on/off)
        in_grid[light_to_switch.row, light_to_switch.col]=light_to_switch.statut
    
    return in_grid

# func_switch_all_lights_statut(in_grid = data, in_lights_to_switch=lights_to_switch)

# Execute une étape

In [15]:
def func_execute_step(
    in_grid: np.array,
    in_dim_grid: int
):
    
    lights_to_switch = func_get_lights_to_switch(
        in_grid = in_grid,
        in_dim_grid = in_dim_grid    
    )

    new_grid = func_switch_all_lights_statut(
        in_grid = in_grid,
        in_lights_to_switch = lights_to_switch    
    )
    
    return new_grid

# Execute toutes les étapes

In [16]:
# print(grid)
for i in range(100):
    grid = func_execute_step(
        in_grid = grid,
        in_dim_grid = dim_grid
    )
    # print(grid)
    # print()

sum(
    1 
    for row in grid    
    for light in row
    if light == "#"
 )

814